# Практика · Самонаглядове навчання

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит передтренує **чотирнадцять** мереж без жодної мітки й навчає ще
> **шість десятків** маленьких класифікаторів. Заміряно на чотирьох ядрах без
> відеокарти: **близько чотирьох з половиною хвилин** на завантаженій машині
> (чистого процесорного часу — дві з половиною). Рахунок іде в один потік:
> так і швидше, і відтворюваніше.

У лекції ми стверджували шість речей. Тут кожна перетворюється на число.

1. **Замір 1 — чи розвʼязний привід.** Порахуємо, для якої частки фігур поворот
   визначається однозначно. Виявиться, що для однієї з шести.
2. **Замір 2 — контрастне передтренування** проти навчання з нуля: таблиця
   «міток × спосіб», три зерна, з розкидом.
3. **Замір 3 — розворот:** на якій кількості міток передтренування перестає вигравати.
4. **Замір 4 — що вивчило тіло** без жодної мітки: точність найближчого сусіда
   в просторі ознак.
5. **Замір 5 — аугментації вирішують:** заберемо одну й подивимось, скільки це коштує.
6. **Замір 6 — скільки непозначених даних треба:** 300, 1200, 4800.

Плюс власна реалізація втрати **NT-Xent** із перевіркою руками на маленькому прикладі.

**Мережа не потрібна:** усі зображення ми малюємо формулами.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)

# один потік, а не чотири. Мережі тут крихітні, і чотири потоки більше часу
# домовляються між собою, ніж рахують. Друга причина важливіша за швидкість:
# під кількома потоками float-суми йдуть в іншому порядку, і числа пливуть.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Привід: задача, у якій мітку дає саме зображення

Самонаглядове навчання починається з **приводу** (pretext task) — вигаданої задачі,
відповідь на яку відома без людини. Класичний приклад: повернути зображення на
0°, 90°, 180° або 270° і вчити мережу вгадувати кут. Кут ми обрали самі, отже
мітка безкоштовна.

Але перш ніж навчати що-небудь, треба відповісти на питання, яке пропускають
майже всі туторіали: **чи взагалі має ця задача відповідь на наших даних?**

Намалюємо шість наших фігур і поруч — ті самі фігури, повернуті на 90°.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.45, center=None, radius=None):
    """Малює одну фігуру як масив 28×28 зі значеннями 0..1.

    center і radius можна задати явно — тоді генератор випадкових чисел не
    потрібен узагалі. Саме так ми зробимо «еталонну» фігуру без шуму й зсуву,
    щоб перевіряти геометрію, а не випадковість.
    """
    image = np.zeros((size, size), dtype=np.float32)
    if center is None:
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    if noise > 0:
        image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


# «еталонні» фігури: рівно по центру, без шуму — тільки геометрія
clean_shapes = [draw_shape(kind, None, center=(13.5, 13.5), radius=7, noise=0.0)
                for kind in range(6)]

import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 6, figsize=(9, 3.4))
for kind in range(6):
    axes[0, kind].imshow(clean_shapes[kind], cmap="gray")
    axes[0, kind].set_title(SHAPE_NAMES[kind], fontsize=9)
    axes[1, kind].imshow(np.rot90(clean_shapes[kind], 1), cmap="gray")
    axes[1, kind].set_title("та сама, 90°", fontsize=8)
    axes[0, kind].axis("off")
    axes[1, kind].axis("off")
plt.tight_layout()
plt.show()
print("верхній рядок — фігура, нижній — вона ж, повернута на 90°")

## 2 · Замір 1: рахуємо, чи розвʼязний привід

Дивитись очима мало — порахуємо. Привід «вгадай кут» має відповідь лише тоді,
коли всі чотири повороти зображення **різні між собою**. Якщо повернута фігура
збігається з вихідною, то правильної відповіді не існує: одна й та сама картинка
є і «0°», і «90°».

Перевіримо три приводи одразу:

- **поворот** — чи всі чотири повороти різні;
- **дзеркало** — чи відрізняється фігура від свого віддзеркалення;
- **квадрант** — чи всі чотири чверті зображення різні між собою
  (привід «вгадай, звідки вирізано патч»).

In [ ]:
def all_different(views):
    """True, якщо всі варіанти попарно різні — тобто відповідь однозначна."""
    for first in range(len(views)):
        for second in range(first + 1, len(views)):
            if np.array_equal(views[first], views[second]):
                return False
    return True


print("%-11s %-12s %-12s %-12s %-12s" %
      ("фігура", "90° = та сама", "180° = та сама", "поворот", "квадрант"))
print("-" * 62)

rotation_solvable = 0
flip_solvable = 0
quadrant_solvable = 0

for kind in range(6):
    image = clean_shapes[kind]
    rotations = [np.rot90(image, k) for k in range(4)]
    quadrants = [image[:14, :14], image[:14, 14:], image[14:, :14], image[14:, 14:]]

    same_after_90 = np.array_equal(image, rotations[1])
    same_after_180 = np.array_equal(image, rotations[2])
    rotation_ok = all_different(rotations)
    flip_ok = not np.array_equal(image, np.fliplr(image))
    quadrant_ok = all_different(quadrants)

    rotation_solvable += rotation_ok
    flip_solvable += flip_ok
    quadrant_solvable += quadrant_ok

    print("%-11s %-12s %-12s %-12s %-12s" %
          (SHAPE_NAMES[kind],
           "так" if same_after_90 else "ні",
           "так" if same_after_180 else "ні",
           "однозначно" if rotation_ok else "НЕМАЄ ВІДПОВІДІ",
           "однозначно" if quadrant_ok else "НЕМАЄ ВІДПОВІДІ"))

print()
print("частка фігур, для яких привід розвʼязний:")
print("  поворот  : %.3f  (%d із 6)" % (rotation_solvable / 6, rotation_solvable))
print("  дзеркало : %.3f  (%d із 6)" % (flip_solvable / 6, flip_solvable))
print("  квадрант : %.3f  (%d із 6)" % (quadrant_solvable / 6, quadrant_solvable))

Ось і відповідь. Привід «вгадай поворот» розвʼязний рівно для **однієї фігури з шести** —
для трикутника. Привід «вгадай дзеркало» не розвʼязний **узагалі ні для однієї**:
всі шість наших фігур симетричні відносно вертикалі. А привід «вгадай квадрант»
розвʼязний для всіх шести — чверті зображення різні навіть у кола, бо чверть кола
з отвором угору-ліворуч не дорівнює чверті з отвором угору-праворуч.

Це не дрібниця й не особливість саме наших фігур. Це **обовʼязкова перевірка перед
будь-яким приводом**: якщо для більшості твоїх даних правильної відповіді не існує,
мережа не вивчить нічого — вона просто знайде обхідний шлях.

## 3 · Датасет

Далі йде наскрізний приклад теми: ті самі шість фігур, але вже з випадковим зсувом
центра (±5 пікселів) і шумом σ = 0.45.

- **4800 непозначених** зображень — на них іде передтренування; міток ми не торкаємось;
- **300 позначених** — з них братимемо 30, 60, 120 або 300 прикладів;
- **600 для перевірки**.

In [ ]:
def make_dataset(count, rng):
    """Повертає (count, 1, 28, 28) і (count,). Класів шість, порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6                       # рівно по шостій частині кожного класу
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
x_unlabeled, _ = make_dataset(4800, rng)      # мітки є, але ми ними НЕ користуємось
x_pool, y_pool = make_dataset(300, rng)
x_test, y_test = make_dataset(600, rng)

print("непозначених :", tuple(x_unlabeled.shape))
print("позначений пул:", tuple(x_pool.shape), " класів:", len(set(y_pool.tolist())))
print("перевірка     :", tuple(x_test.shape))
print("рівень вгадування на шести класах: %.3f" % (1 / 6))

## 4 · Дешевий обхід: чому «різні пікселі» не означає «розвʼязно»

Тепер важлива пастка. Якщо взяти сирі зображення датасету — зі зсувом і шумом — і
спитати, чи відрізняється зображення від свого повороту, відповідь буде «так»
практично завжди. Шум і зсув центра роблять будь-які дві картинки різними.

Але це **не** означає, що привід став осмисленим. Мережа справді зможе вгадувати
кут — за зсувом центра й за візерунком шуму, а не за формою фігури. Вона навчиться
рівно нічого корисного. Порахуємо цю оманливу частку.

In [ ]:
rotation_looks_solvable = 0
for i in range(300):
    image = x_unlabeled[i, 0].numpy()
    if not np.array_equal(image, np.rot90(image, 1)):
        rotation_looks_solvable += 1

print("на сирих зображеннях поворот «помітний» у %.3f випадків"
      % (rotation_looks_solvable / 300))
print("на чистій геометрії фігури — лише в %.3f" % (rotation_solvable / 6))
print()
print("різниця між цими двома числами і є та дірка, у яку провалюється привід:")
print("мережа розвʼязує задачу за шумом і зсувом, а не за формою.")

## 5 · Мережа: тіло й голова

Мережа та сама, що в темі «Перенос навчання»: три блоки `Conv → ReLU → Pool`,
які перетворюють 28×28 на 288 чисел, і голова, яка ухвалює рішення.

Нове тут одне — **проєктор** (projection head). Під час контрастного навчання ми
порівнюємо не самі 288 чисел, а їхнє стиснення до 32. Так робить SimCLR, і причина
проста: втрата тягне свій вихід у дуже специфічний бік, і краще, щоб вона робила
це з окремим шматком мережі, який ми потім викинемо. Лишиться тіло — воно й
цінне.

In [ ]:
def conv_block(in_channels, out_channels):
    """Один типовий блок: Conv → ReLU → Pool."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class Body(nn.Module):
    """Тіло: 1×28×28 → 288 чисел (32 канали по 3×3)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(1, 8),        # 1×28×28  →  8×14×14
            conv_block(8, 16),       # 8×14×14  → 16×7×7
            conv_block(16, 32),      # 16×7×7   → 32×3×3
            nn.Flatten(),
        )

    def forward(self, x):
        return self.net(x)


class Projector(nn.Module):
    """Тимчасова надбудова для контрастної втрати: 288 → 64 → 32. Потім викидається."""

    def __init__(self, dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(288, 64), nn.ReLU(), nn.Linear(64, dim))

    def forward(self, x):
        return self.net(x)


class ShapeNet(nn.Module):
    """Тіло плюс голова — звичайний класифікатор."""

    def __init__(self, n_classes=6):
        super().__init__()
        self.body = Body()
        self.head = nn.Linear(288, n_classes)

    def forward(self, x):
        return self.head(self.body(x))


torch.manual_seed(0)
probe_model = ShapeNet(6)
body_params = sum(p.numel() for p in probe_model.body.parameters())
head_params = sum(p.numel() for p in probe_model.head.parameters())

# голова — це рівно 288×6 ваг плюс 6 зсувів; порахуємо руками й звіримо
head_by_hand = 288 * 6 + 6
assert head_params == head_by_hand, "рахунок голови розійшовся!"

print("параметрів у тілі  :", body_params)
print("параметрів у голові:", head_params, " рукою:", head_by_hand, "✅ збігається")
print("тіло тримає %.0f%% усіх ваг" % (100 * body_params / (body_params + head_params)))

## 6 · Чому датасет саме такий складний

Одна деталь, без якої весь замір був би порожнім. Наші фігури можна зробити
чистішими — шум 0.20 замість 0.45, зсув ±4 замість ±5. І тоді вимірювати буде
нічого: **навіть випадкове, ніяк не навчене тіло** розділяє класи майже добре.

Перевіримо це прямо. Візьмемо непідготовлену мережу з випадковими вагами,
прогонимо через неї зображення й подивимось, чи достатньо близько лежать сусіди
одного класу. Міра — точність найближчого сусіда (`kNN`) у просторі ознак:
для кожного тестового зображення шукаємо найсхожіше з відомих і беремо його клас.

In [ ]:
@torch.no_grad()
def knn_accuracy(body, x_bank, y_bank, x_query, y_query):
    """Точність найближчого сусіда в просторі ознак — навчання з мітками НЕ потрібне.

    Ознаки нормуємо на довжину 1, тоді скалярний добуток — це косинус кута між
    ними, тобто пряма міра «наскільки два зображення схожі очима мережі».
    """
    body.eval()
    bank = F.normalize(body(x_bank), dim=1)
    query = F.normalize(body(x_query), dim=1)
    similarity = query @ bank.t()
    nearest = similarity.argmax(dim=1)
    return (y_bank[nearest] == y_query).float().mean().item()


def make_easy_dataset(count, rng):
    """Той самий генератор, але з мʼякшими налаштуваннями: шум 0.20, зсув ±4."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng, jitter=4, noise=0.20)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


easy_rng = np.random.default_rng(7)
easy_pool_x, easy_pool_y = make_easy_dataset(300, easy_rng)
easy_test_x, easy_test_y = make_easy_dataset(600, easy_rng)

torch.manual_seed(7)
easy_knn = knn_accuracy(Body(), easy_pool_x, easy_pool_y, easy_test_x, easy_test_y)
torch.manual_seed(7)
hard_knn = knn_accuracy(Body(), x_pool, y_pool, x_test, y_test)

print("kNN на ВИПАДКОВОМУ (не навченому) тілі:")
print("  легкі фігури (шум 0.20, зсув ±4): %.3f" % easy_knn)
print("  наші фігури  (шум 0.45, зсув ±5): %.3f" % hard_knn)
print("  рівень вгадування               : %.3f" % (1 / 6))
print()
print("на легких фігурах випадкова згортка вже майже розвʼязує задачу —")
print("передтренуванню не лишилось би що покращувати. Тому беремо складніші.")

## 7 · Два види одного зображення

Контрастне навчання не вгадує нічого про зображення. Воно робить простішу річ:
бере одне зображення, псує його **двічі по-різному** й вимагає, щоб мережа впізнала
в цих двох виглядах те саме.

Аугментації тут — не косметика, а **сама постановка задачі**. Кожна аугментація —
це заява «оце міняти можна, клас від цього не змінюється». Тому серед них немає
повороту: ми щойно порахували, що поворот наших фігур не міняє, і зближувати
повернуті види означало б вчити мережу ігнорувати те, чого вона й так не бачить.

Беремо три: **зсув** (±3 пікселі), **яскравість і контраст**, **додатковий шум**.

In [ ]:
def augment(x, generator, shift=True, brightness=True, noise=True, max_shift=3):
    """Випадковий вигляд партії зображень. Жодна операція не міняє клас фігури."""
    n = x.shape[0]
    out = x

    if shift:
        # доповнюємо кадр нулями й вирізаємо вікно 28×28 у випадковому місці
        padded = F.pad(out, (max_shift, max_shift, max_shift, max_shift))
        offset_y = torch.randint(0, 2 * max_shift + 1, (n,), generator=generator)
        offset_x = torch.randint(0, 2 * max_shift + 1, (n,), generator=generator)
        moved = torch.empty_like(out)
        for i in range(n):
            moved[i] = padded[i, :, offset_y[i]:offset_y[i] + 28,
                              offset_x[i]:offset_x[i] + 28]
        out = moved

    if brightness:
        # множник міняє контраст, доданок — загальну яскравість
        gain = 0.6 + 0.8 * torch.rand(n, 1, 1, 1, generator=generator)
        bias = -0.15 + 0.3 * torch.rand(n, 1, 1, 1, generator=generator)
        out = out * gain + bias

    if noise:
        out = out + 0.15 * torch.randn(out.shape, generator=generator)

    return out.clamp(0, 1)


views_generator = torch.Generator().manual_seed(5)
sample = x_pool[:6]
view_one = augment(sample, views_generator)
view_two = augment(sample, views_generator)

figure, axes = plt.subplots(3, 6, figsize=(9, 4.8))
for slot in range(6):
    axes[0, slot].imshow(sample[slot, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, slot].imshow(view_one[slot, 0], cmap="gray", vmin=0, vmax=1)
    axes[2, slot].imshow(view_two[slot, 0], cmap="gray", vmin=0, vmax=1)
    for row in range(3):
        axes[row, slot].axis("off")
axes[0, 0].set_title("оригінал", fontsize=8, loc="left")
axes[1, 0].set_title("вид 1", fontsize=8, loc="left")
axes[2, 0].set_title("вид 2", fontsize=8, loc="left")
plt.tight_layout()
plt.show()
print("два нижні рядки — той самий предмет. Мережа має це зрозуміти сама.")

## 8 · NT-Xent: пишемо втрату самі й перевіряємо руками

Тепер головна формула теми. У партії з N зображень ми зробили 2N видів. Для
кожного виду є рівно **один свій** (другий вид того самого зображення) і 2N−2
чужих. Втрата вимагає: свій має бути схожішим за всіх чужих.

Для одного виду з номером i:

&nbsp;&nbsp;&nbsp;&nbsp;втрата(i) = −log( exp(s(i, свій)/τ) / Σ<sub>j≠i</sub> exp(s(i, j)/τ) )

де s(a, b) — косинусна схожість двох векторів (від −1 до 1), τ — температура,
а Σ (грецька сигма) означає «додай усе, що йде далі». У чисельнику стоїть **свій**,
у знаменнику — **усі, крім себе самого**. Мінус логарифм перетворює «хочу, щоб
частка була близька до 1» на «хочу, щоб число було близьке до 0».

Спершу напишемо це матрицями, потім порахуємо той самий вираз у лоб, циклом, і
звіримо. Якщо збіглося — реалізація правильна.

In [ ]:
def nt_xent(z_one, z_two, temperature=0.1):
    """Контрастна втрата NT-Xent для двох наборів видів однієї партії."""
    n = z_one.shape[0]

    # нормуємо на довжину 1: після цього скалярний добуток дорівнює косинусу кута
    z = F.normalize(torch.cat([z_one, z_two], dim=0), dim=1)

    similarity = z @ z.t() / temperature
    # себе з собою не порівнюємо: інакше «свій» завжди програвав би самому собі
    similarity.fill_diagonal_(-1e9)

    # для виду i з першої половини свій — це i+n, для виду з другої — i−n
    partner = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)])
    return F.cross_entropy(similarity, partner)


# ---- перевірка руками на партії з трьох зображень ----
torch.manual_seed(0)
demo_one = torch.randn(3, 4)
demo_two = torch.randn(3, 4)
library_value = nt_xent(demo_one, demo_two, temperature=0.5)

# той самий вираз у лоб, без жодних матричних трюків
demo_z = F.normalize(torch.cat([demo_one, demo_two], dim=0), dim=1)
total = 0.0
for i in range(6):
    partner_index = i + 3 if i < 3 else i - 3
    numerator = torch.exp(demo_z[i] @ demo_z[partner_index] / 0.5)
    denominator = 0.0
    for j in range(6):
        if j != i:
            denominator = denominator + torch.exp(demo_z[i] @ demo_z[j] / 0.5)
    total = total + (-torch.log(numerator / denominator))
by_hand_value = total / 6

print("наша реалізація : %.6f" % library_value.item())
print("рахунок руками  : %.6f" % by_hand_value.item())
assert torch.allclose(library_value, by_hand_value, atol=1e-6), "розрахунок розійшовся!"
print("✅ збігається")

# і ще одна перевірка сенсу: якщо два види ідентичні, втрата має бути малою
identical = torch.randn(8, 4)
print()
print("втрата, коли види збігаються ідеально: %.3f" % nt_xent(identical, identical, 0.1).item())
print("втрата на випадкових векторах        : %.3f"
      % nt_xent(torch.randn(8, 4), torch.randn(8, 4), 0.1).item())
print("рівень «нічого не знаю» = log(2N−1) = %.3f" % np.log(2 * 8 - 1))

## 9 · Передтренування без жодної мітки

Тепер сам цикл. Він нічим не відрізняється від звичайного навчання, крім одного:
`y` тут немає взагалі. Партію беремо, робимо два види, рахуємо NT-Xent, крок.

Партія навмисно велика — 256. Контрастна втрата тим змістовніша, чим більше в
знаменнику чужих прикладів: при партії 8 мережі треба відрізнити свій вид від
14 чужих, при 256 — від 510.

Три зерна, бо одне число нічого не варте без розкиду.

In [ ]:
PRETRAIN_EPOCHS = 4
BATCH = 256
TEMPERATURE = 0.1
SEEDS = (0, 1, 2)


def pretrain_contrastive(x_unlabeled_part, epochs=PRETRAIN_EPOCHS, batch=BATCH,
                         lr=3e-3, seed=0, temperature=TEMPERATURE, aug_kwargs=None):
    """Вчить тіло без міток: два види одного зображення мають зійтися."""
    if aug_kwargs is None:
        aug_kwargs = {}
    torch.manual_seed(seed)
    body = Body()
    projector = Projector()
    optimizer = torch.optim.Adam(list(body.parameters()) + list(projector.parameters()),
                                 lr=lr)
    generator = torch.Generator().manual_seed(seed + 1000)

    losses = []
    count = len(x_unlabeled_part)
    body.train()
    for _ in range(epochs):
        order = torch.randperm(count, generator=generator)
        epoch_loss, steps = 0.0, 0
        for start in range(0, count, batch):
            index = order[start:start + batch]
            if len(index) < 8:                # надто мала партія знаменник не наповнить
                continue
            images = x_unlabeled_part[index]
            first = augment(images, generator, **aug_kwargs)
            second = augment(images, generator, **aug_kwargs)
            optimizer.zero_grad()
            loss = nt_xent(projector(body(first)), projector(body(second)), temperature)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            steps += 1
        losses.append(epoch_loss / max(steps, 1))
    return body, losses


contrastive_bodies = {}
print("передтренування: епох %d, партія %d, температура %.2f, непозначених %d"
      % (PRETRAIN_EPOCHS, BATCH, TEMPERATURE, len(x_unlabeled)))
print("рівень «нічого не знаю» для втрати = log(2·%d−1) = %.3f" % (BATCH, np.log(2 * BATCH - 1)))
print()
for seed in SEEDS:
    started = time.perf_counter()
    body, losses = pretrain_contrastive(x_unlabeled, seed=seed)
    contrastive_bodies[seed] = body
    print("зерно %d: %4.0f с   втрата %.3f → %.3f   kNN тіла %.3f"
          % (seed, time.perf_counter() - started, losses[0], losses[-1],
             knn_accuracy(body, x_pool, y_pool, x_test, y_test)))

## 10 · Той самий бюджет, але привід «вгадай поворот»

Щоб порівняння було чесним, повторимо все з нерозвʼязним приводом: ті самі 4800
непозначених зображень, ті самі епохи, ті самі зерна. Мережа отримує повернуте
зображення й має назвати кут — задача з чотирма відповідями, вгадування дає 0.25.

In [ ]:
def pretrain_rotation(x_unlabeled_part, epochs=PRETRAIN_EPOCHS, batch=BATCH, lr=3e-3, seed=0):
    """Вироджений привід: вгадай кут повороту 0/90/180/270."""
    torch.manual_seed(seed)
    body = Body()
    head = nn.Linear(288, 4)
    optimizer = torch.optim.Adam(list(body.parameters()) + list(head.parameters()), lr=lr)
    generator = torch.Generator().manual_seed(seed + 1000)

    count = len(x_unlabeled_part)
    accuracies = []
    body.train()
    for _ in range(epochs):
        order = torch.randperm(count, generator=generator)
        right, total = 0, 0
        for start in range(0, count, batch):
            index = order[start:start + batch]
            if len(index) < 8:
                continue
            images = x_unlabeled_part[index]
            turns = torch.randint(0, 4, (len(index),), generator=generator)
            # мітка береться з самої операції: ми ж і вирішили, на скільки крутити
            turned = torch.stack([torch.rot90(images[i], int(turns[i]), dims=(1, 2))
                                  for i in range(len(index))])
            optimizer.zero_grad()
            logits = head(body(turned))
            loss = F.cross_entropy(logits, turns)
            loss.backward()
            optimizer.step()
            right += (logits.argmax(1) == turns).sum().item()
            total += len(index)
        accuracies.append(right / max(total, 1))
    return body, accuracies


rotation_bodies = {}
print("привід «вгадай поворот», вгадування дає 0.250")
print()
for seed in SEEDS:
    started = time.perf_counter()
    body, accuracies = pretrain_rotation(x_unlabeled, seed=seed)
    rotation_bodies[seed] = body
    print("зерно %d: %4.0f с   точність приводу %.3f → %.3f   kNN тіла %.3f"
          % (seed, time.perf_counter() - started, accuracies[0], accuracies[-1],
             knn_accuracy(body, x_pool, y_pool, x_test, y_test)))

## 11 · Замір 4: що вивчило тіло, поки не бачило жодної мітки

Найпряміше питання теми. У нас є тіло, яке ніколи не бачило слова «трикутник».
Чи розкладає воно фігури по різних кутках свого простору ознак?

Перевірка не вимагає жодного навчання: беремо 300 зображень із відомими класами
як довідник, для кожного тестового шукаємо найсхожіше й дивимось, чи збігся клас.
Порівнюємо три тіла: контрастне, «поворотне» і **зовсім випадкове**.

Випадкове тіло тут — не жарт, а обовʼязкова точка відліку. Згорткова мережа з
випадковими вагами вже щось робить із зображенням, і без цього рядка легко
приписати передтренуванню чужу заслугу.

In [ ]:
knn_results = {}
for name, bodies in (("контрастне", contrastive_bodies), ("поворот", rotation_bodies)):
    values = [knn_accuracy(bodies[seed], x_pool, y_pool, x_test, y_test) for seed in SEEDS]
    knn_results[name] = (float(np.mean(values)), float(np.std(values)))

random_values = []
for seed in SEEDS:
    torch.manual_seed(100 + seed)
    random_values.append(knn_accuracy(Body(), x_pool, y_pool, x_test, y_test))
knn_results["випадкове"] = (float(np.mean(random_values)), float(np.std(random_values)))

print("%-14s %s" % ("тіло", "kNN у просторі ознак (3 зерна)"))
print("-" * 46)
for name in ("випадкове", "поворот", "контрастне"):
    mean, spread = knn_results[name]
    print("%-14s %.3f ±%.3f" % (name, mean, spread))
print("%-14s %.3f" % ("вгадування", 1 / 6))

Подивимось на це очима. Візьмемо 288 чисел, які тіло видає на 180 зображеннях,
і стиснемо їх до двох вимірів методом головних компонент (PCA — він шукає два
напрямки, уздовж яких хмара точок розтягнута найбільше). Точки фарбуємо
**справжнім класом**, якого тіло не бачило. Три панелі — три тіла: випадкове,
після нерозвʼязного приводу й після контрастного навчання.

Заразом надрукуємо, яку частку всього розкиду вміщують ці дві осі. Тут чекає
пастка, на яку варто подивитись самому: більша частка **не** означає кращого
представлення.

In [ ]:
def pca_two_dimensions(features):
    """Дві головні компоненти плюс частка розкиду, яку вони пояснюють."""
    centered = features - features.mean(axis=0)
    _, singular, directions = np.linalg.svd(centered, full_matrices=False)
    coordinates = centered @ directions[:2].T
    explained = float((singular[:2] ** 2).sum() / (singular ** 2).sum())
    return coordinates, explained


with torch.no_grad():
    contrastive_bodies[0].eval()
    contrastive_features = contrastive_bodies[0](x_pool[:180]).numpy()
    rotation_bodies[0].eval()
    rotation_features = rotation_bodies[0](x_pool[:180]).numpy()
    torch.manual_seed(100)
    untrained_body = Body()
    untrained_body.eval()
    random_features = untrained_body(x_pool[:180]).numpy()

contrastive_xy, contrastive_share = pca_two_dimensions(contrastive_features)
rotation_xy, rotation_share = pca_two_dimensions(rotation_features)
random_xy, random_share = pca_two_dimensions(random_features)
true_labels = y_pool[:180].numpy()

figure, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for axis, coordinates, title in ((axes[0], random_xy, "випадкове тіло"),
                                 (axes[1], rotation_xy, "після приводу «поворот»"),
                                 (axes[2], contrastive_xy, "після контрастного навчання")):
    for kind in range(6):
        mask = true_labels == kind
        axis.scatter(coordinates[mask, 0], coordinates[mask, 1], s=16,
                     label=SHAPE_NAMES[kind])
    axis.set_title(title, fontsize=10)
    axis.set_xlabel("перша головна компонента", fontsize=8)
    axis.set_ylabel("друга головна компонента", fontsize=8)
axes[2].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

print("дві компоненти пояснюють розкиду:")
print("  випадкове тіло : %.1f %%" % (100 * random_share))
print("  привід поворот : %.1f %%" % (100 * rotation_share))
print("  контрастне     : %.1f %%" % (100 * contrastive_share))
print()
print("велика частка сама собою нічого не варта: у «поворотного» тіла вона найбільша,")
print("а класи в ньому не розділяються (kNN %.3f проти %.3f у контрастного)."
      % (knn_results["поворот"][0], knn_results["контрастне"][0]))

## 12 · Три способи скористатися тілом

Далі — механіка з теми «Перенос навчання», без змін:

| Спосіб | Тіло | Що вчиться |
|---|---|---|
| З нуля | випадкове | усе |
| Лінійна проба | передтреноване, заморожене | лише лінійний шар 288 → 6 |
| Донавчання | передтреноване, вільне | усе |

Лінійна проба — головна міра якості представлення. Якщо класи розділяються
**однією прямою** в просторі ознак, значить тіло вже зробило найважчу частину роботи.

In [ ]:
def train_classifier(model, x, y, epochs=25, lr=3e-3, batch=8, seed=0, params=None):
    """Звичайне навчання з мітками. params=None означає «вчити все, що є»."""
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(
        params if params is not None else model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(x))
        for start in range(0, len(x), batch):
            index = order[start:start + batch]
            optimizer.zero_grad()
            loss = loss_function(model(x[index]), y[index])
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def accuracy(model, x, y):
    model.eval()
    return (model(x).argmax(1) == y).float().mean().item()


def train_from_scratch(x, y, seed=0):
    """Жодного знання ззовні: тіло випадкове, вчиться все."""
    torch.manual_seed(seed)
    model = ShapeNet(6)
    train_classifier(model, x, y, seed=seed)
    return accuracy(model, x_test, y_test)


def linear_probe(body_state, x, y, seed=0):
    """Тіло заморожене, вчиться лише лінійний шар — міра якості представлення."""
    torch.manual_seed(seed)
    model = ShapeNet(6)
    model.body.load_state_dict(body_state)
    for parameter in model.body.parameters():
        parameter.requires_grad = False
    train_classifier(model, x, y, seed=seed, params=list(model.head.parameters()))
    return accuracy(model, x_test, y_test)


def finetune(body_state, x, y, seed=0):
    """Тіло передтреноване, але вільне: вчиться все."""
    torch.manual_seed(seed)
    model = ShapeNet(6)
    model.body.load_state_dict(body_state)
    train_classifier(model, x, y, seed=seed)
    return accuracy(model, x_test, y_test)


torch.manual_seed(0)
demo = ShapeNet(6)
for parameter in demo.body.parameters():
    parameter.requires_grad = False
print("вільних чисел при лінійній пробі :",
      sum(p.numel() for p in demo.parameters() if p.requires_grad))
print("вільних чисел при донавчанні     :", sum(p.numel() for p in ShapeNet(6).parameters()))

## 13 · Заміри 2 і 3: таблиця «міток × спосіб»

Головний замір теми. Чотири кількості міток, чотири способи, три зерна.
Це найдовша клітинка зошита — **близько півтори хвилини**.

In [ ]:
LABEL_COUNTS = (30, 60, 120, 300)
results = {}

print("%-7s %-16s %-16s %-16s %-16s"
      % ("міток", "з нуля", "проба (контр.)", "донавч. (контр.)", "проба (поворот)"))
print("-" * 76)

for count in LABEL_COUNTS:
    x_part, y_part = x_pool[:count], y_pool[:count]
    row = {}

    values = [train_from_scratch(x_part, y_part, seed=s) for s in SEEDS]
    row["scratch"] = (float(np.mean(values)), float(np.std(values)))

    values = [linear_probe(contrastive_bodies[s].state_dict(), x_part, y_part, seed=s)
              for s in SEEDS]
    row["probe"] = (float(np.mean(values)), float(np.std(values)))

    values = [finetune(contrastive_bodies[s].state_dict(), x_part, y_part, seed=s)
              for s in SEEDS]
    row["finetune"] = (float(np.mean(values)), float(np.std(values)))

    values = [linear_probe(rotation_bodies[s].state_dict(), x_part, y_part, seed=s)
              for s in SEEDS]
    row["rotation_probe"] = (float(np.mean(values)), float(np.std(values)))

    results[count] = row
    print("%-7d %.3f ±%.3f     %.3f ±%.3f     %.3f ±%.3f     %.3f ±%.3f"
          % (count, *row["scratch"], *row["probe"], *row["finetune"], *row["rotation_probe"]))

print()
print("рівень вгадування: %.3f" % (1 / 6))

Тепер витягнемо з таблиці головне: **де проходить розворот**. Порівняємо лінійну
пробу з навчанням з нуля на кожній кількості міток.

In [ ]:
print("%-7s %-10s %-10s %-12s" % ("міток", "з нуля", "проба", "хто виграв"))
print("-" * 44)
crossover = None
for count in LABEL_COUNTS:
    scratch = results[count]["scratch"][0]
    probe = results[count]["probe"][0]
    winner = "проба" if probe > scratch else "з нуля"
    if probe <= scratch and crossover is None:
        crossover = count
    print("%-7d %-10.3f %-10.3f %-12s (різниця %+.3f)"
          % (count, scratch, probe, winner, probe - scratch))

print()
print("самонаглядове передтренування перестає вигравати на %d мітках" % crossover)
print()
print("а ось що дав нерозвʼязний привід (поворот) — порівняно з навчанням з нуля:")
for count in LABEL_COUNTS:
    print("  %3d міток: поворот %.3f проти %.3f з нуля  (%+.3f)"
          % (count, results[count]["rotation_probe"][0], results[count]["scratch"][0],
             results[count]["rotation_probe"][0] - results[count]["scratch"][0]))

## 14 · Замір 5: аугментації — це і є задача

Контрастна мережа вчиться **рівно тому, що ти оголосив неістотним**. Заберемо
зсув із набору аугментацій. Тоді два види одного зображення відрізнятимуться лише
яскравістю й шумом, а фігура лишатиметься на місці — і мережі стане вигідно
впізнавати види за **положенням**, а не за формою.

Для контролю заберемо натомість яскравість. Два зерна на конфігурацію.

In [ ]:
ablation = {}
ablation["повний набір"] = (results[60]["probe"][0], results[30]["probe"][0],
                            knn_results["контрастне"][0])

print("%-18s %-14s %-14s %s" % ("набір аугментацій", "проба, 60", "проба, 30", "kNN тіла"))
print("-" * 62)

for name, options in (("без зсуву", dict(shift=False)),
                      ("без яскравості", dict(brightness=False))):
    probe_60, probe_30, knn_values = [], [], []
    for seed in SEEDS[:2]:
        body, _ = pretrain_contrastive(x_unlabeled, seed=seed, aug_kwargs=options)
        state = body.state_dict()
        knn_values.append(knn_accuracy(body, x_pool, y_pool, x_test, y_test))
        probe_60.append(linear_probe(state, x_pool[:60], y_pool[:60], seed=seed))
        probe_30.append(linear_probe(state, x_pool[:30], y_pool[:30], seed=seed))
    ablation[name] = (float(np.mean(probe_60)), float(np.mean(probe_30)),
                      float(np.mean(knn_values)))

for name in ("повний набір", "без зсуву", "без яскравості"):
    print("%-18s %-14.3f %-14.3f %.3f" % (name, *ablation[name]))

## 15 · Замір 6: скільки непозначених даних треба

Головний аргумент самонаглядового навчання — непозначених даних можна взяти
скільки завгодно. Перевіримо, чи справді від їхньої кількості щось залежить:
передтренуємо на 300, 1200 і 4800 зображеннях і щоразу зміряємо лінійну пробу
на 60 мітках. Два зерна.

In [ ]:
volume = {}
print("%-16s %-16s %s" % ("непозначених", "проба, 60 міток", "kNN тіла"))
print("-" * 46)

for size in (300, 1200):
    probe_values, knn_values = [], []
    for seed in SEEDS[:2]:
        body, _ = pretrain_contrastive(x_unlabeled[:size], seed=seed)
        knn_values.append(knn_accuracy(body, x_pool, y_pool, x_test, y_test))
        probe_values.append(linear_probe(body.state_dict(), x_pool[:60], y_pool[:60],
                                         seed=seed))
    volume[size] = (float(np.mean(probe_values)), float(np.mean(knn_values)))
    print("%-16d %-16.3f %.3f" % (size, *volume[size]))

volume[4800] = (results[60]["probe"][0], knn_results["контрастне"][0])
print("%-16d %-16.3f %.3f" % (4800, *volume[4800]))
print()
print("на 60 мітках навчання з нуля дає %.3f" % results[60]["scratch"][0])

## 16 · Підсумок числами

In [ ]:
print("=" * 62)
print("ЩО ВИЙШЛО")
print("=" * 62)
print("1. привід «поворот» розвʼязний для %.3f фігур, «дзеркало» для %.3f, «квадрант» для %.3f"
      % (rotation_solvable / 6, flip_solvable / 6, quadrant_solvable / 6))
print("2. тіло без жодної мітки: kNN %.3f проти %.3f у випадкового тіла"
      % (knn_results["контрастне"][0], knn_results["випадкове"][0]))
print("3. нерозвʼязний привід дає kNN %.3f — не краще за випадкове тіло"
      % knn_results["поворот"][0])
print("4. лінійна проба виграє в навчання з нуля на %d мітках (%.3f проти %.3f)"
      % (30, results[30]["probe"][0], results[30]["scratch"][0]))
print("5. і програє на %d (%.3f проти %.3f)"
      % (crossover, results[crossover]["probe"][0], results[crossover]["scratch"][0]))
print("6. прибрати зсув з аугментацій коштує %.3f точності проби на 30 мітках"
      % (ablation["повний набір"][1] - ablation["без зсуву"][1]))
print("7. учетверо більше непозначених даних (300 → 1200): проба %.3f → %.3f"
      % (volume[300][0], volume[1200][0]))
print()
print("зошит виконувався %.0f с" % (time.perf_counter() - notebook_started))

## Завдання

### 🟢 Рівень 1
Зміни температуру τ у `nt_xent` на 0.5 і передтренуй одне тіло. Порівняй kNN
із тим, що вийшло при 0.1. **Зроблено, якщо** ти назвав обидва числа й пояснив
словами, що робить температура зі знаменником.

### 🟡 Рівень 2
Додай до набору аугментацій стирання випадкового квадрата 8×8 (`RandomErasing`
з теми «Аугментація» або просто присвоєння нулів). Передтренуй і зміряй пробу.
**Зроблено, якщо** ти пояснив отримане число через розмір фігури: радіус у нас
5-8 пікселів, а квадрат 8×8.

### 🔴 Рівень 3
Вигадай свій привід — наприклад, «вгадай, у скільки разів зображення зменшили»
або «вгадай, який із чотирьох квадрантів вирізано». **Спершу доведи числом**,
що він розвʼязний на наших фігурах (як у розділі 2), і лише потім навчай.
**Зроблено, якщо** є і частка розвʼязних випадків, і таблиця «проба проти
навчання з нуля» на 30 і 60 мітках.